In [3]:
import os
import re
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from dotenv import load_dotenv

load_dotenv(override=True)
model = ChatOpenAI(model="gpt-5.4-mini")

INTERVIEW_SYSTEM = """당신은 사용자가 만들고 싶은 'SKILL.md' 명세를 끌어내는 인터뷰어입니다.

목표: 사용자가 말한 스킬 아이디어에서 다음 항목들이 모두 명확해질 때까지 한 번에 1~3개씩 핵심 질문을 던지세요.
필수 확인 항목:
1. 스킬의 이름과 한 줄 요약 (description)
2. 어떤 상황/요청에서 이 스킬을 발동해야 하는가 (트리거)
3. 단계별 절차 (1단계, 2단계 …)
4. 각 단계에서 LLM이 절대 놓치면 안 되는 체크 항목
5. 결과를 어떤 형식/말투로 출력해야 하는가
6. 예외 상황 처리 방법

규칙:
- 사용자의 답이 모호하면 더 구체적으로 파고드세요. ("그 '검토'가 정확히 뭘 보는 건가요?")
- 이미 답한 내용은 다시 묻지 마세요.
- 사용자의 분야/맥락을 추측하지 말고 직접 물어보세요.
- 한국어로 답하세요.
- **응답 마지막에는 반드시 다음 문장을 그대로 출력하세요:**
  지금까지의 대화내역으로 SKILL을 만들고자 하시면 '만족'이라고 입력하세요.
"""

GENERATE_PROMPT = """지금까지의 인터뷰 대화를 바탕으로 완성된 SKILL.md 파일 본문을 생성하세요.

반드시 아래 형식을 그대로 따르세요. 코드펜스(```)나 추가 설명 없이 SKILL.md 본문만 출력하세요.

---
name: <소문자-하이픈-식별자>
description: <한 문장으로 이 스킬이 언제 쓰이는지>
---

# <스킬 제목>

## 사용 시기
- ...

## 절차
### 1단계. ...
- ...

### 2단계. ...
- ...

(필요한 만큼 단계를 추가)

## 출력 형식
...

## 예외 처리
- ...
"""

# 인터뷰 루프
history = [SystemMessage(content=INTERVIEW_SYSTEM)]

print("=" * 60)
print("[SKILL 만들기 인터뷰] 만들고 싶은 스킬에 대해 자유롭게 말씀해주세요.")
print("(종료하려면 '만족' 입력 / 강제 중단은 Ctrl+C)")
print("=" * 60)

while True:
    user_input = input("\n나> ").strip()
    if not user_input:
        continue

    if user_input == "만족":
        print("\n[인터뷰 종료] SKILL.md 생성 중...")
        break

    history.append(HumanMessage(content=user_input))
    ai_msg = model.invoke(history)
    history.append(AIMessage(content=ai_msg.content))
    print(f"\nLLM> {ai_msg.content}")

# SKILL.md 생성
gen_messages = history + [HumanMessage(content=GENERATE_PROMPT)]
skill_md = model.invoke(gen_messages).content.strip()

# 코드펜스가 섞여 들어와도 제거
skill_md = re.sub(r"^```(?:markdown|md)?\s*|\s*```$", "", skill_md, flags=re.MULTILINE).strip()

# name 추출해서 폴더명으로 사용
match = re.search(r"^name:\s*(\S+)", skill_md, re.MULTILINE)
skill_name = match.group(1) if match else "my-skill"

skill_dir = f"./skills/{skill_name}"
os.makedirs(skill_dir, exist_ok=True)
skill_path = f"{skill_dir}/SKILL.md"

with open(skill_path, "w", encoding="utf-8") as f:
    f.write(skill_md)

print("\n" + "=" * 60)
print(f"SKILL.md 생성 완료: {skill_path}")
print("=" * 60)
print(skill_md)


[SKILL 만들기 인터뷰] 만들고 싶은 스킬에 대해 자유롭게 말씀해주세요.
(종료하려면 '만족' 입력 / 강제 중단은 Ctrl+C)



LLM> 좋습니다. 지금 의도는 “도시 + 여행 기간(몇 박 몇 일)”만 받으면 바로 여행 계획을 생성하는 스킬로 보입니다.  
다만 SKILL.md로 정확히 쓰려면 아래 3가지만 더 확인해야 합니다.

1. **출력 형식**
   - 여행 계획을 어떤 형식으로 만들까요?
   - 예: 하루별 일정표, 아침/점심/저녁 포함, 표 형식, bullet 형식

2. **계획의 범위**
   - 무엇을 포함해야 하나요?
   - 예: 관광지, 맛집, 이동 동선, 숙소 추천, 예산, 교통수단

3. **예외 상황 처리**
   - 도시명이나 박수/일수가 너무 모호하면 어떻게 할까요?
   - 예: “파리 3박4일”처럼만 주면 바로 짜고, 정보가 부족하면 최소한 1번만 추가 질문

지금까지의 대화내역으로 SKILL을 만들고자 하시면 '만족'이라고 입력하세요.

LLM> 좋습니다. 이제 거의 명확합니다. 추가로 2가지만 확인하면 SKILL 명세를 바로 정리할 수 있습니다.

1. **하루별 구성**
   - 각 날짜마다 어떻게 배치할까요?
   - 예: 오전 관광지 / 점심 맛집 / 오후 관광지 / 저녁 맛집 / 숙소

2. **출력 톤과 기준**
   - 추천은 어떤 톤으로 할까요?
   - 예: 실용적이고 간결하게, 아니면 친절하게 이유까지 설명
   - 숙소 추천은 어떤 기준을 우선할까요?
   - 예: 가성비, 위치, 교통 편의, 분위기

지금까지의 대화내역으로 SKILL을 만들고자 하시면 '만족'이라고 입력하세요.

[인터뷰 종료] SKILL.md 생성 중...

SKILL.md 생성 완료: ./skills/travel-plan-generator/SKILL.md
---
name: travel-plan-generator
description: 사용자가 도시와 몇 박 몇 일을 입력하면 관광지, 맛집, 숙소 추천을 포함한 여행 계획을 바로 생성할 때 사용합니다.
---

# 여행 계획 생성기

## 사용 시기
- 사용자가 도시와 여행 기간(몇 박 몇 일)을 입력했을 